# Insmile AI — Fine-Tune Qwen2-VL-7B for Dental X-Ray Analysis

This notebook fine-tunes **Qwen2-VL-7B** on the **DENTEX** dataset using **QLoRA** (4-bit quantization + LoRA adapters) to fit within Google Colab's free T4 GPU (16GB VRAM).

**What this produces:** A LoRA adapter (~150MB) that makes the model significantly better at:
- Detecting dental caries, deep caries, periapical lesions, and impacted teeth
- Providing accurate FDI tooth numbers
- Generating precise bounding boxes
- Outputting structured JSON

## Instructions
1. Open this in Google Colab (GPU runtime → T4)
2. Run all cells in order
3. Training takes ~3-5 hours on a free T4
4. The adapter auto-uploads to HuggingFace when done

---

## 1. Setup — Install Dependencies

In [ ]:
%%capture
!pip install -U transformers accelerate bitsandbytes peft trl datasets
!pip install -U qwen-vl-utils pillow
!pip install -U huggingface_hub
!pip install -U scipy
print('All dependencies installed.')

## 2. Configuration

In [ ]:
import os
from google.colab import userdata

# ============================================================
# CONFIGURATION — Edit these values
# ============================================================

# HuggingFace token — set in Colab secrets (Key icon on left sidebar → add HF_TOKEN)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError(
        'HF_TOKEN not found! Add it in Colab: Left sidebar → Key icon → '
        'New secret → Name: HF_TOKEN, Value: your hf_... token'
    )

# Model
BASE_MODEL = 'Qwen/Qwen2-VL-7B-Instruct'  # Instruct version for better JSON output

# Where to save the fine-tuned adapter on HuggingFace
HF_REPO_NAME = 'joshuarebo/insmile-dental-vision-lora'

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 1              # Limited by VRAM
GRAD_ACCUM_STEPS = 8       # Effective batch size = 1 * 8 = 8
LEARNING_RATE = 2e-4       # Standard for QLoRA
MAX_SEQ_LENGTH = 2048      # Max tokens per example
LORA_R = 16                # LoRA rank (higher = more capacity, more VRAM)
LORA_ALPHA = 32            # LoRA scaling factor
LORA_DROPOUT = 0.05

print(f'Model: {BASE_MODEL}')
print(f'Output repo: {HF_REPO_NAME}')
print(f'Epochs: {EPOCHS}, LR: {LEARNING_RATE}, LoRA rank: {LORA_R}')

## 3. Login to HuggingFace

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

## 4. Download DENTEX Dataset

In [ ]:
import subprocess
import json
import os
from pathlib import Path

DATASET_DIR = Path('/content/dentex')
DATASET_DIR.mkdir(exist_ok=True)

# Try loading via HuggingFace datasets library (handles images automatically)
try:
    from datasets import load_dataset
    print('Downloading DENTEX from HuggingFace...')
    dentex_ds = load_dataset('ibrahimethemhamamci/DENTEX', split='train', trust_remote_code=True)
    USE_HF_DATASET = True
    print(f'Loaded {len(dentex_ds)} examples from HuggingFace')
    print(f'Columns: {dentex_ds.column_names}')
    if len(dentex_ds) > 0:
        print(f'First example keys: {list(dentex_ds[0].keys())}')
        for k, v in dentex_ds[0].items():
            print(f'  {k}: {type(v).__name__} = {str(v)[:200]}')
except Exception as e:
    print(f'HuggingFace dataset load failed: {e}')
    print('Falling back to GitHub clone + git-lfs...')
    USE_HF_DATASET = False

    if not (DATASET_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/ibrahimethemhamamci/DENTEX.git',
                        str(DATASET_DIR)], check=True)

    subprocess.run(['apt-get', 'install', '-y', 'git-lfs'], capture_output=True)
    subprocess.run(['git', 'lfs', 'install'], cwd=str(DATASET_DIR), capture_output=True)
    subprocess.run(['git', 'lfs', 'pull'], cwd=str(DATASET_DIR))

    image_files = list(DATASET_DIR.rglob('*.png')) + list(DATASET_DIR.rglob('*.jpg'))
    json_files = list(DATASET_DIR.rglob('*.json'))
    print(f'Found {len(image_files)} images and {len(json_files)} JSON files')
    for d in sorted(set(p.parent for p in image_files[:20])):
        count = len(list(d.glob('*.png'))) + len(list(d.glob('*.jpg')))
        print(f'  {d.relative_to(DATASET_DIR)}: {count} images')

## 5. Parse DENTEX Annotations → Training Format

We convert the COCO-format annotations into instruction-tuning examples:
- **Input**: dental X-ray image + analysis prompt
- **Output**: structured JSON with findings, tooth numbers, severity, bounding boxes

In [ ]:
import json
import glob
from pathlib import Path
from PIL import Image

# DENTEX category mapping
DENTEX_CATEGORIES = {
    1: {'label': 'Caries', 'severity': 'moderate'},
    2: {'label': 'Deep caries', 'severity': 'severe'},
    3: {'label': 'Periapical lesion', 'severity': 'severe'},
    4: {'label': 'Impacted tooth', 'severity': 'moderate'},
}

# FDI quadrant mapping for tooth number estimation from bbox position
def estimate_fdi_from_bbox(bbox_norm, img_width, img_height):
    """Estimate FDI tooth number from bbox position on a panoramic X-ray.
    Panoramic X-rays are mirrored: patient's right is on image left.
    """
    cx = bbox_norm[0] + bbox_norm[2] / 2  # center x (0-1)
    cy = bbox_norm[1] + bbox_norm[3] / 2  # center y (0-1)
    
    # Determine quadrant
    is_upper = cy < 0.5
    is_right_side = cx < 0.5  # Image left = patient's right
    
    if is_upper and is_right_side:
        quadrant = 1
    elif is_upper and not is_right_side:
        quadrant = 2
    elif not is_upper and not is_right_side:
        quadrant = 3
    else:
        quadrant = 4
    
    # Estimate tooth position (1-8) based on distance from midline
    dist_from_center = abs(cx - 0.5) * 2  # 0 = midline, 1 = edge
    tooth_num = min(8, max(1, int(dist_from_center * 8) + 1))
    
    return f'{quadrant}{tooth_num}'


def parse_dentex_annotations(json_path, images_dir):
    """Parse COCO-format DENTEX annotations into training examples."""
    with open(json_path) as f:
        data = json.load(f)
    
    # Build image lookup
    images = {img['id']: img for img in data.get('images', [])}
    
    # Build category lookup
    categories = {}
    for cat in data.get('categories', []):
        cat_id = cat['id']
        cat_name = cat['name'].lower()
        if 'deep' in cat_name or 'caries' in cat_name and 'deep' in cat_name:
            categories[cat_id] = {'label': 'Deep caries', 'severity': 'severe'}
        elif 'caries' in cat_name:
            categories[cat_id] = {'label': 'Caries', 'severity': 'moderate'}
        elif 'periapical' in cat_name:
            categories[cat_id] = {'label': 'Periapical lesion', 'severity': 'severe'}
        elif 'impacted' in cat_name:
            categories[cat_id] = {'label': 'Impacted tooth', 'severity': 'moderate'}
        else:
            categories[cat_id] = {'label': cat['name'], 'severity': 'moderate'}
    
    # If no categories in file, use defaults
    if not categories:
        categories = DENTEX_CATEGORIES
    
    # Group annotations by image
    img_annotations = {}
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        if img_id not in img_annotations:
            img_annotations[img_id] = []
        img_annotations[img_id].append(ann)
    
    examples = []
    for img_id, anns in img_annotations.items():
        if img_id not in images:
            continue
        
        img_info = images[img_id]
        img_path = images_dir / img_info['file_name']
        if not img_path.exists():
            # Try common alternatives
            for alt in [images_dir / Path(img_info['file_name']).name,
                        images_dir.parent / 'xrays' / img_info['file_name']]:
                if alt.exists():
                    img_path = alt
                    break
            else:
                continue
        
        img_w = img_info.get('width', 1900)
        img_h = img_info.get('height', 950)
        
        findings = []
        for ann in anns:
            cat_id = ann.get('category_id', 1)
            cat = categories.get(cat_id, {'label': 'Abnormality', 'severity': 'moderate'})
            
            # COCO bbox format: [x, y, width, height] in pixels
            bbox = ann.get('bbox', [0, 0, 100, 100])
            bbox_norm = [
                round(bbox[0] / img_w, 4),
                round(bbox[1] / img_h, 4),
                round(bbox[2] / img_w, 4),
                round(bbox[3] / img_h, 4),
            ]
            
            # Estimate tooth number from position
            tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
            
            findings.append({
                'label': f"{cat['label']} on tooth {tooth}",
                'tooth': tooth,
                'severity': cat['severity'],
                'confidence': 0.92,
                'bbox_norm': bbox_norm,
            })
        
        if not findings:
            continue
        
        # Build the target JSON
        target = {
            'findings': findings[:6],
            'overall': f"Panoramic X-ray showing {len(findings)} pathological finding{'s' if len(findings) > 1 else ''}: {', '.join(set(c['label'].split(' on ')[0] for c in findings[:4]))}.",
            'confidence': 0.88,
            'recommendations': generate_recommendations(findings),
            'image_quality': 'good',
        }
        
        examples.append({
            'image_path': str(img_path),
            'target_json': json.dumps(target, indent=None),
        })
    
    return examples


def generate_recommendations(findings):
    """Generate clinical recommendations based on findings."""
    recs = []
    labels = [f['label'].split(' on ')[0].lower() for f in findings]
    
    if 'deep caries' in labels:
        recs.append('Urgent: Root canal treatment or extraction may be needed for deep caries')
    if 'caries' in labels:
        recs.append('Composite or amalgam restoration recommended for carious lesions')
    if 'periapical lesion' in labels:
        recs.append('Periapical pathology detected — consider endodontic evaluation and vitality testing')
    if 'impacted tooth' in labels:
        recs.append('Surgical evaluation recommended for impacted tooth — assess proximity to IAN canal')
    if not recs:
        recs.append('Regular follow-up and preventive care recommended')
    
    recs.append('Patient education on oral hygiene and dietary habits')
    return recs[:4]


print('Annotation parser ready.')

In [ ]:
# Build training examples from whatever source loaded successfully
all_examples = []

if USE_HF_DATASET:
    print('Parsing HuggingFace dataset...')

    for i, item in enumerate(dentex_ds):
        try:
            img = item.get('image')
            if img is None:
                continue

            # Save image to disk for training pipeline
            img_path = DATASET_DIR / f'image_{i:04d}.png'
            if not img_path.exists():
                img.save(str(img_path))

            img_w, img_h = img.size
            findings = []

            # Try common DENTEX HF schema patterns
            annotations = item.get('annotations', item.get('objects', item.get('label', None)))

            if annotations and isinstance(annotations, dict):
                bboxes = annotations.get('bbox', annotations.get('bboxes', []))
                cats = annotations.get('category', annotations.get('categories',
                       annotations.get('label', [])))

                if isinstance(bboxes, list) and len(bboxes) > 0:
                    if isinstance(bboxes[0], (list, tuple)):
                        bbox_list = bboxes
                    else:
                        if len(bboxes) == 4 and all(isinstance(b, (int, float)) for b in bboxes):
                            bbox_list = [bboxes]
                        else:
                            bbox_list = [bboxes[j:j+4] for j in range(0, len(bboxes), 4)]

                    if not isinstance(cats, list):
                        cats = [cats] * len(bbox_list)

                    for bbox, cat_id in zip(bbox_list, cats):
                        if len(bbox) < 4:
                            continue
                        bbox_norm = [
                            round(bbox[0] / img_w, 4) if bbox[0] > 1 else round(bbox[0], 4),
                            round(bbox[1] / img_h, 4) if bbox[1] > 1 else round(bbox[1], 4),
                            round(bbox[2] / img_w, 4) if bbox[2] > 1 else round(bbox[2], 4),
                            round(bbox[3] / img_h, 4) if bbox[3] > 1 else round(bbox[3], 4),
                        ]

                        cat_map = {0: 'Caries', 1: 'Deep caries', 2: 'Periapical lesion',
                                   3: 'Impacted tooth', 'Caries': 'Caries',
                                   'Deep Caries': 'Deep caries',
                                   'Periapical Lesion': 'Periapical lesion',
                                   'Impacted': 'Impacted tooth'}
                        label = cat_map.get(cat_id, f'Abnormality (class {cat_id})')
                        severity = 'severe' if 'deep' in label.lower() or 'periapical' in label.lower() else 'moderate'

                        tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
                        findings.append({
                            'label': f'{label} on tooth {tooth}',
                            'tooth': tooth,
                            'severity': severity,
                            'confidence': 0.92,
                            'bbox_norm': bbox_norm,
                        })

            elif annotations and isinstance(annotations, list):
                for ann in annotations:
                    if isinstance(ann, dict):
                        bbox = ann.get('bbox', [0, 0, 100, 100])
                        cat_name = ann.get('category', ann.get('label', 'Abnormality'))
                    else:
                        continue
                    bbox_norm = [
                        round(bbox[0] / img_w, 4) if bbox[0] > 1 else round(bbox[0], 4),
                        round(bbox[1] / img_h, 4) if bbox[1] > 1 else round(bbox[1], 4),
                        round(bbox[2] / img_w, 4) if bbox[2] > 1 else round(bbox[2], 4),
                        round(bbox[3] / img_h, 4) if bbox[3] > 1 else round(bbox[3], 4),
                    ]
                    tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
                    severity = 'severe' if 'deep' in str(cat_name).lower() or 'periapical' in str(cat_name).lower() else 'moderate'
                    findings.append({
                        'label': f'{cat_name} on tooth {tooth}',
                        'tooth': tooth,
                        'severity': severity,
                        'confidence': 0.92,
                        'bbox_norm': bbox_norm,
                    })

            if not findings:
                continue

            target = {
                'findings': findings[:6],
                'overall': f"Panoramic X-ray showing {len(findings)} pathological finding{'s' if len(findings) > 1 else ''}: {', '.join(set(f['label'].split(' on ')[0] for f in findings[:4]))}.",
                'confidence': 0.88,
                'recommendations': generate_recommendations(findings),
                'image_quality': 'good',
            }

            all_examples.append({
                'image_path': str(img_path),
                'target_json': json.dumps(target, indent=None),
            })
        except Exception as ex:
            if i < 5:
                print(f'  Skipped example {i}: {ex}')
            continue

    print(f'\nParsed {len(all_examples)} examples from HuggingFace dataset')

else:
    # Parse from cloned GitHub repo
    json_files = list(DATASET_DIR.rglob('*.json'))
    print(f'Found {len(json_files)} JSON files in cloned repo')

    for jf in json_files:
        try:
            with open(jf) as f:
                data = json.load(f)
            if 'annotations' not in data or 'images' not in data:
                continue
        except:
            continue

        images_dir = None
        for candidate in [jf.parent, jf.parent / 'xrays', jf.parent / 'images',
                          jf.parent.parent / 'xrays', jf.parent.parent / 'images',
                          jf.parent.parent / 'training', DATASET_DIR / 'images']:
            if candidate and candidate.exists():
                imgs = list(candidate.glob('*.png')) + list(candidate.glob('*.jpg'))
                if imgs:
                    images_dir = candidate
                    break

        if not images_dir:
            print(f'  No images found for {jf.relative_to(DATASET_DIR)}')
            continue

        examples = parse_dentex_annotations(jf, images_dir)
        if examples:
            print(f'  {jf.relative_to(DATASET_DIR)}: {len(examples)} examples')
            all_examples.extend(examples)

if not all_examples:
    print('\n⚠️  No examples parsed! Debug info:')
    if USE_HF_DATASET and len(dentex_ds) > 0:
        print(f'Dataset has {len(dentex_ds)} rows but none parsed.')
        print(f'Columns: {dentex_ds.column_names}')
        print('First 3 rows:')
        for idx in range(min(3, len(dentex_ds))):
            print(f'\n--- Row {idx} ---')
            for k, v in dentex_ds[idx].items():
                print(f'  {k} ({type(v).__name__}): {str(v)[:300]}')
    else:
        print('Listing files in dataset dir:')
        for p in sorted(DATASET_DIR.rglob('*'))[:50]:
            if p.is_file():
                print(f'  {p.relative_to(DATASET_DIR)} ({p.stat().st_size/1024:.0f}KB)')
    raise RuntimeError('No training examples. Check debug output above — send it to me and I will fix the parser.')

print(f'\nTotal training examples: {len(all_examples)}')
sample = all_examples[0]
print(f'Sample image: {sample["image_path"]}')
print(f'Sample target:\n{sample["target_json"][:300]}')

## 6. Build Training Dataset

Convert to the chat format that Qwen2-VL expects for instruction tuning.

In [ ]:
import random
from datasets import Dataset

# System prompt (same as our production prompt, so the model learns our exact format)
SYSTEM_PROMPT = """You are an expert dental radiologist AI. Analyze the dental X-ray and return a JSON object with your findings.

Output format:
{"findings": [{"label": "specific finding", "tooth": "FDI number", "severity": "mild|moderate|severe", "confidence": 0.0-1.0, "bbox_norm": [x, y, w, h]}], "overall": "summary", "confidence": 0.0-1.0, "recommendations": ["action"], "image_quality": "good|fair|poor"}

Rules: bbox_norm values are 0.0-1.0 (normalized). Use FDI tooth numbering. Return JSON ONLY."""

USER_PROMPT = "Analyze this dental radiograph. Identify all visible pathology using FDI tooth numbering. Return ONLY the JSON object."

def build_conversation(example):
    """Build a chat conversation for training."""
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': example['image_path']},
                    {'type': 'text', 'text': USER_PROMPT},
                ]
            },
            {'role': 'assistant', 'content': example['target_json']},
        ]
    }

# Shuffle and split
random.seed(42)
random.shuffle(all_examples)

split_idx = max(1, int(len(all_examples) * 0.9))
train_examples = all_examples[:split_idx]
val_examples = all_examples[split_idx:] if split_idx < len(all_examples) else all_examples[-1:]

train_conversations = [build_conversation(ex) for ex in train_examples]
val_conversations = [build_conversation(ex) for ex in val_examples]

print(f'Training examples: {len(train_conversations)}')
print(f'Validation examples: {len(val_conversations)}')

# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_conversations)
val_dataset = Dataset.from_list(val_conversations)

print(f'\nDataset ready.')
if len(train_dataset) > 0:
    print(f'Sample user prompt: {train_dataset[0]["messages"][1]["content"][1]}')
    print(f'Sample target (first 200 chars): {train_dataset[0]["messages"][2]["content"][:200]}')

## 7. Load Model with QLoRA (4-bit Quantization)

This loads the 7B model in 4-bit precision (~4GB VRAM) and attaches trainable LoRA adapters.

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading model in 4-bit...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print(f'Model loaded. Parameters: {model.num_parameters():,}')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# LoRA configuration — attach adapters to attention layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention
        'gate_proj', 'up_proj', 'down_proj',       # MLP
    ],
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'GPU memory after LoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 8. Training with SFTTrainer

Using TRL's SFTTrainer which handles the vision-language chat format automatically.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/content/insmile-dental-checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field='',           # Not used for vision
    dataset_kwargs={'skip_prepare_dataset': True},
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to='none',
)

print('Training configuration ready.')
print(f'  Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'  Total training steps: ~{len(train_conversations) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}')

In [ ]:
from functools import partial
from qwen_vl_utils import process_vision_info

def collate_fn(examples, processor):
    """Custom collator for vision-language training."""
    texts = []
    image_inputs = []
    
    for example in examples:
        messages = example['messages']
        # Apply chat template
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        
        # Process images
        images, videos = process_vision_info(messages)
        image_inputs.append(images)
    
    # Tokenize
    batch = processor(
        text=texts,
        images=image_inputs[0] if image_inputs[0] else None,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors='pt',
    )
    
    # Labels = input_ids (causal LM training)
    batch['labels'] = batch['input_ids'].clone()
    
    return batch

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=partial(collate_fn, processor=processor),
    processing_class=processor,
)

print('Trainer initialized. Ready to train.')

In [ ]:
# ============================================================
# TRAIN! This takes 3-5 hours on a T4 GPU.
# ============================================================
print('Starting training...')
print('='*60)

train_result = trainer.train()

print('='*60)
print('Training complete!')
print(f'  Loss: {train_result.training_loss:.4f}')
print(f'  Runtime: {train_result.metrics["train_runtime"]/3600:.1f} hours')
print(f'  Samples/sec: {train_result.metrics["train_samples_per_second"]:.2f}')

## 9. Save & Upload Adapter to HuggingFace

In [ ]:
# Save the LoRA adapter locally
ADAPTER_PATH = '/content/insmile-dental-adapter'
model.save_pretrained(ADAPTER_PATH)
processor.save_pretrained(ADAPTER_PATH)

print(f'Adapter saved to {ADAPTER_PATH}')

# Check adapter size
import os
total_size = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f))
                 for f in os.listdir(ADAPTER_PATH)
                 if os.path.isfile(os.path.join(ADAPTER_PATH, f)))
print(f'Adapter size: {total_size / 1e6:.1f} MB')

In [ ]:
# Upload to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(HF_REPO_NAME, private=True, exist_ok=True)
except Exception as e:
    print(f'Repo creation note: {e}')

# Upload adapter
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=HF_REPO_NAME,
    commit_message='Insmile dental vision LoRA adapter - trained on DENTEX',
)

print(f'\nAdapter uploaded to: https://huggingface.co/{HF_REPO_NAME}')
print('\nYou can now use this adapter in your Insmile backend!')

## 10. Quick Validation — Test the Fine-Tuned Model

In [ ]:
# Test on a validation image
if val_examples:
    test_example = val_examples[0]
    test_image = Image.open(test_example['image_path']).convert('RGB')
    
    test_messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': test_image},
                {'type': 'text', 'text': USER_PROMPT},
            ]
        },
    ]
    
    text = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    images, _ = process_vision_info(test_messages)
    
    inputs = processor(
        text=[text], images=images, return_tensors='pt', padding=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1000, temperature=0.1)
    
    response = processor.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    
    print('MODEL OUTPUT:')
    print(response[:500])
    print('\n---\nEXPECTED:')
    print(test_example['target_json'][:500])
else:
    print('No validation examples available for testing.')

## Done!

Your fine-tuned LoRA adapter is now on HuggingFace. Next steps:

1. **Deploy**: Host the model on RunPod Serverless or HuggingFace Inference Endpoints
2. **Integrate**: Update `server/src/services/openrouter.js` to call your self-hosted model
3. **Iterate**: As dentists use the app and correct findings, save those corrections as new training data

---
*Generated by Insmile AI training pipeline*